# Numerical simulation 4: estuarine circulation

**37EFM Environmental Fluid Mechanics, TU/e**

A channel of depth $H = 15$ m connects a river to the sea. Three things drive the flow, and all three are set in `efmgotm/cases/estuary.yaml` under `mimic_3d`:

| forcing | key | default |
|---|---|---|
| river outflow, as a mean depth-averaged velocity $q/H$ [m/s] | `mimic_3d.ext_pressure.dpdx.constant_value` | $-0.02$ |
| tide, amplitude of the depth-averaged velocity [m/s] | `mimic_3d.ext_pressure.dpdx.tidal.amp_1` | $0.5$ |
| tidal period [s] | `mimic_3d.ext_pressure.period_1` | 44714 |
| along-channel salinity gradient [psu/m] | `mimic_3d.int_pressure.gradients.dsdx.constant_value` | $-2 \times 10^{-4}$ |

The salinity gradient gives a horizontal density gradient, hence an internal pressure gradient that varies with depth. Salinity is advected by the flow and relaxed to 15 psu on the tidal time scale so that it stays bounded. The equation of state is linear with $\beta = 7.7 \times 10^{-4}$ psu$^{-1}$ and $\rho_0 = 1027$ kg/m$^3$. The run covers 29 days with output every 6 minutes.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import efmgotm
from efmgotm import analysis
plt.rcParams['figure.dpi'] = 80
RUNS = 'runs'          # run directories are created here

In [ ]:
ds = efmgotm.run('estuary', workdir=RUNS)
H = float(-ds.zi.isel(zi=0))
M2 = 44714.0
ds

## (a) Which tidal constituent is forced?

The spectrum of the depth-averaged velocity tells. The M2 period is 12.42 h, S2 is 12.00 h, K1 and O1 are near 24 h.

In [ ]:
u_bar = analysis.depth_average(ds.u, ds)
f, p = analysis.spectrum(u_bar)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
u_bar.plot(ax=ax1, x='days'); ax1.set_ylabel('depth-averaged u [m/s]'); ax1.set_xlim(0, 3)
ax2.loglog(1 / f[1:] / 3600, p[1:])
for period, name in [(12.42, 'M2'), (12.0, 'S2'), (6.21, 'M4'), (24.0, 'diurnal')]:
    ax2.axvline(period, color='gray', ls=':', lw=0.8); ax2.text(period, p[1:].max(), name, fontsize=8, rotation=90, va='top')
ax2.set_xlabel('period [h]'); ax2.set_ylabel('power'); ax2.set_xlim(1, 100);

## (b) The net volume flux

The river outflow is the tidal average of the depth-integrated velocity, $q = \langle \int_{-H}^0 u\,dz \rangle$ in m$^2$/s (per unit width). Use the last tidal cycle, after the spin-up.

In [ ]:
u_int = (ds.u * ds.h).sum('z')                       # depth-integrated velocity [m^2/s]
q = float(analysis.tidal_average(u_int, M2))
print(f'q = {q:.3f} m^2/s   (q/H = {q / H:.4f} m/s)')

## Profiles over a tidal cycle

Velocity, salinity and turbulent viscosity at several phases of the last tidal cycle. The flood is the landward phase, the ebb the seaward one. Which direction is seaward here?

In [ ]:
t_end = float(ds.seconds[-1])
phases = np.linspace(0, 1, 8, endpoint=False)
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
colors = plt.cm.twilight(phases)
for ph, c in zip(phases, colors):
    sel = ds.sel(seconds=t_end - M2 + ph * M2, method='nearest')
    axs[0].plot(sel.u, sel.z, color=c, label=f'{ph:.3f} T')
    axs[1].plot(sel.salt, sel.z, color=c)
    axs[2].semilogx(sel.num, sel.zi, color=c)
axs[0].set_xlabel('u [m/s]'); axs[0].set_ylabel('z [m]'); axs[0].legend(fontsize=8, title='phase')
axs[1].set_xlabel('S [psu]'); axs[2].set_xlabel(r'$\nu_t$ [m$^2$/s]'); axs[2].set_xlim(1e-6, 1e-1);

In [ ]:
# the last two tidal cycles as colour maps
last = ds.isel(time=slice(-int(2 * M2 / 360), None))
fig, axs = plt.subplots(3, 1, figsize=(10, 11), sharex=True)
last.u.plot(ax=axs[0], x='hours', y='z', cmap='RdBu_r')
last.salt.plot(ax=axs[1], x='hours', y='z', cmap='viridis')
np.log10(last.num).plot(ax=axs[2], x='hours', y='zi', cmap='magma', vmin=-5, vmax=-1, cbar_kwargs={'label': r'$\log_{10}\nu_t$'})
axs[0].set_title('u'); axs[1].set_title('salinity'); axs[2].set_title('turbulent viscosity');

## (c) The tidally averaged circulation

The theoretical profile on the sheet is

$$u = \frac{3q}{2H}\left(1 - \left[\frac{z}{H}\right]^2\right) - \frac{H^3}{48 A}\frac{\partial b}{\partial x}\left(1 - 9\left[\frac{z}{H}\right]^2 - 8\left[\frac{z}{H}\right]^3\right),$$

with $\partial b/\partial x = -g\beta\,\partial s/\partial x$ and a constant eddy viscosity $A$. Compare it with the tidal average of the model, and find a reasonable $A$. The model's own viscosity, tidally averaged, is plotted alongside.

In [ ]:
dsdx = -2e-4
dbdx = analysis.buoyancy_gradient(dsdx, beta=7.7e-4)
u_avg = analysis.tidal_average(ds.u, M2)
num_avg = analysis.tidal_average(ds.num, M2)
print(f'db/dx = {dbdx:.2e} s^-2')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
ax1.plot(u_avg, ds.z, 'k', lw=2, label='GOTM, tidal average')
for A in [1e-3, 3e-3, 1e-2]:
    ax1.plot(analysis.estuarine_profile(ds.z.values, H, q, dbdx, A), ds.z, '--', label=f'theory, A = {A:g} m$^2$/s')
ax1.axvline(0, color='gray', lw=0.5)
ax1.set_xlabel('u [m/s]'); ax1.set_ylabel('z [m]'); ax1.legend(fontsize=8)
ax2.plot(num_avg, ds.zi); ax2.set_xlabel(r'tidally averaged $\nu_t$ [m$^2$/s]');

## Changing the forcing

Everything in the table at the top can be changed. For instance, without the tide the circulation is driven by the river and the density gradient alone:

In [ ]:
ds_notide = efmgotm.run('estuary', {'mimic_3d.ext_pressure.dpdx.tidal.amp_1': 0.0, 'time.stop': '2005-01-11 00:00:00'},
                        name='no_tide', workdir=RUNS)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(u_avg, ds.z, label='with tide, tidal average')
ax.plot(ds_notide.u.isel(time=-1), ds_notide.z, label='no tide, day 10')
ax.axvline(0, color='gray', lw=0.5); ax.set_xlabel('u [m/s]'); ax.set_ylabel('z [m]'); ax.legend();